In [24]:
import numpy as np
import plotly.graph_objects as go
from scipy.integrate import solve_ivp

def lorenz_system(t, state, sigma=10, rho=28, beta=8/3):
    """Lorenz 63 system equations"""
    x, y, z = state
    dx = sigma * (y - x)
    dy = x * (rho - z) - y
    dz = x * y - beta * z
    return [dx, dy, dz]

def generate_lorenz_trajectory(dt=0.01, t_max=100, initial_state=[1, 1, 1]):
    """Generate Lorenz 63 trajectory using numerical integration"""
    t_span = (0, t_max)
    t_eval = np.arange(0, t_max, dt)
    
    sol = solve_ivp(lorenz_system, t_span, initial_state, 
                    t_eval=t_eval, method='RK45', dense_output=True)
    
    return sol.t, sol.y[0], sol.y[1], sol.y[2]

def plot_lorenz_with_windows_interactive(t, x, y, z, windows):
    """
    Create interactive 3D plot of Lorenz 63 trajectory with highlighted windows
    
    Parameters:
    -----------
    t : array
        Time points
    x, y, z : arrays
        Trajectory coordinates
    windows : list of dicts
        Each dict should have: {'start': int, 'length': int, 'color': str, 'label': str}
        Example: [{'start': 2000, 'length': 200, 'color': 'red', 'label': 'Window 1'}]
    """
    
    fig = go.Figure()
    
    # Add full trajectory (thin, semi-transparent)
    fig.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode='lines',
        line=dict(color='steelblue', width=2),
        opacity=0.5,
        name='Full trajectory',
        hoverinfo='skip'
    ))
    
    # Add highlighted windows
    for i, window in enumerate(windows):
        start = window['start']
        length = window['length']
        end = start + length
        color = window.get('color', f'rgb({(i*80)%255}, {(i*120)%255}, {(i*160)%255})')
        label = window.get('label', f'Window {i+1}')
        
        # Add thicker line for the window
        fig.add_trace(go.Scatter3d(
            x=x[start:end], 
            y=y[start:end], 
            z=z[start:end],
            mode='lines',
            line=dict(color=color, width=8),
            name=label,
            hovertemplate=f'<b>{label}</b><br>X: %{{x:.2f}}<br>Y: %{{y:.2f}}<br>Z: %{{z:.2f}}<extra></extra>'
        ))
        
        # Add start point marker
        fig.add_trace(go.Scatter3d(
            x=[x[start]], 
            y=[y[start]], 
            z=[z[start]],
            mode='markers',
            marker=dict(size=3, color=color, symbol='circle',
                       line=dict(color='black', width=2)),
            name=f'{label} - Start',
            hovertemplate=f'<b>{label} Start</b><br>Index: {start}<br>Time: {t[start]:.2f}<extra></extra>'
        ))
        
        # Add end point marker
        fig.add_trace(go.Scatter3d(
            x=[x[end-1]], 
            y=[y[end-1]], 
            z=[z[end-1]],
            mode='markers',
            marker=dict(size=3, color=color, symbol='square',
                       line=dict(color='black', width=2)),
            name=f'{label} - End',
            hovertemplate=f'<b>{label} End</b><br>Index: {end-1}<br>Time: {t[end-1]:.2f}<extra></extra>'
        ))
    
    # Update layout for better interactivity
    fig.update_layout(
        title=dict(
            text='Lorenz 63 System - Interactive Windowed Trajectory',
            font=dict(size=20)
        ),
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            xaxis=dict(gridcolor='white', showbackground=True, backgroundcolor='rgb(256, 256, 256)'),
            yaxis=dict(gridcolor='white', showbackground=True, backgroundcolor='rgb(256, 256, 256)'),
            zaxis=dict(gridcolor='white', showbackground=True, backgroundcolor='rgb(256, 256, 256)'),
            camera=dict(
                eye=dict(x=1.5, y=1.5, z=1.3)
            )
        ),
        width=1200,
        height=800,
        showlegend=False,
        legend=dict(
            x=0.02,
            y=0.98,
            bgcolor='rgba(255, 255, 255, 0.8)',
            bordercolor='black',
            borderwidth=0.1
        ),
        hovermode='closest'
    )
    
    return fig


# Example usage
if __name__ == "__main__":
    # Generate Lorenz 63 trajectory
    print("Generating Lorenz 63 trajectory...")
    t, x, y, z = generate_lorenz_trajectory(dt=0.01, t_max=100)
    print(f"Generated {len(t)} points")
    
    # Define windows to highlight
    windows = [
        {'start': 2000, 'length': 20, 'color': 'blue', 'label': 'Window 1'},
        {'start': 4000, 'length': 20, 'color': 'darkblue', 'label': 'Window 2'},
        {'start': 6000, 'length': 20, 'color': 'navy', 'label': 'Window 3'},
        {'start': 8000, 'length': 20, 'color': 'gold', 'label': 'Window 4'},
        {'start': 9000, 'length': 20, 'color': 'gold', 'label': 'Window 5'}


    ]
    
    # Create interactive plot
    print("Creating interactive visualization...")
    fig = plot_lorenz_with_windows_interactive(t, x, y, z, windows)
    
    # Show the plot
    fig.show()
    
    # Optional: Save as HTML
    # fig.write_html('lorenz_interactive.html')
    # print("Interactive plot saved as 'lorenz_interactive.html'")


# ========== USAGE WITH YOUR DATA ==========
# If you already have your trajectory data:
#
# # Load your data (adjust based on your format)
# # data = np.load('your_data.npy')  # or pd.read_csv, etc.
# # t = data[:, 0]
# # x = data[:, 1]
# # y = data[:, 2]
# # z = data[:, 3]
#
# # Define your windows (indices based on your timestep)
# my_windows = [
#     {'start': 1000, 'length': 500, 'color': 'crimson', 'label': 'Event A'},
#     {'start': 3500, 'length': 300, 'color': 'purple', 'label': 'Event B'},
# ]
#
# # Plot
# fig = plot_lorenz_with_windows_interactive(t, x, y, z, my_windows)
# fig.show()
# 
# # Or save as standalone HTML file
# fig.write_html('my_lorenz_plot.html')

Generating Lorenz 63 trajectory...
Generated 10000 points
Creating interactive visualization...
